# 01a · EDA — 컴바인 (combine_results, combine_tracking)

**목적**: Stage 2(컴바인 특징 추출) 전에 데이터 구조를 확인하고, 다음 단계가 쓸 **기준값을 근거와 함께 확정**한다.

| 입력 | 산출물 |
|---|---|
| `combine_results.csv`, `combine_tracking.csv`, `players.csv`(드래프트 정보만) | ① `config.py` 결정값(글리치 임계값, 트라이얼 키, truncated 기준) ② `reports/tables/1_*.csv` ③ `reports/figures/1_*.png` ④ 8절 요약 → `reports/eda.md` |

이 노트북은 `data/processed/`에 아무것도 쓰지 않는다.

## 0. 환경 설정

In [ ]:
import os, sys
from pathlib import Path

try:                                    # Colab
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/nfl-bdb2027')
except ImportError:                     # 로컬 VS Code (notebooks/에서 실행)
    ROOT = Path.cwd().parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from src import config
config.setup()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

POS = config.POSITION_GROUPS            # ["DB", "OL", "DL", "WR", "TE"]
# 그림 안 글자는 영어로 둔다 (Colab·Mac 모두 한글 폰트 설정 없이 깨지지 않게)


def save_fig(fig, name):
    path = config.fig_path(1, name)
    fig.savefig(path, dpi=config.FIG_DPI, bbox_inches="tight")
    print(f"saved → {path.relative_to(ROOT)}")


def save_table(df, name, index=True):
    path = config.TABLES_DIR / f"1_{name}.csv"
    df.to_csv(path, index=index)
    print(f"saved → {path.relative_to(ROOT)}")

## 1. combine_results 개요
데이터 프로파일(11/15) 기준: 510행, DB 122 / OL 121 / DL 118 / WR 107 / TE 42. 값이 다르면 원본이 바뀐 것이므로 먼저 확인한다.

In [ ]:
cr = pd.read_csv(config.RAW_FILES["combine_results"])
print(cr.shape, "| nfl_id 중복:", cr["nfl_id"].duplicated().sum())
display(cr["combine_position"].value_counts().reindex(POS).to_frame("n"))
display(cr.head())

In [ ]:
# 측정치 그룹 (실제 컬럼명 기준 — 계획서 T2의 height/weight/hand/arm/wingspan 에 대응)
BODY  = ["combine_height", "combine_weight", "hand_size", "arm_length", "wing_span"]
DRILL = ["ten_yd_split", "forty", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps"]
NGS   = ["ngs_athleticism_score", "ngs_college_production_score", "ngs_final_score"]
MEAS  = BODY + DRILL                     # 전통 측정치 (FS-A 후보)

summary = cr.groupby("combine_position")[MEAS].agg(["mean", "std"]).reindex(POS).round(2).T
summary

## 2. 포지션별 결측 패턴
3-cone · short shuttle · bench 결측 60~65%가 **어떤 포지션에서** 생기는지, 그리고 **드래프트 순위와 관련 있는지**(상위 지명자가 드릴을 건너뛰는 MNAR 가능성)를 본다. Stage 4 결측 처리 비교 실험의 설계 근거.

In [ ]:
miss = (cr.groupby("combine_position")[MEAS + NGS]
          .apply(lambda d: d.isna().mean())
          .reindex(POS))
miss.loc["ALL"] = cr[MEAS + NGS].isna().mean()

fig, ax = plt.subplots(figsize=(11, 3.6))
sns.heatmap(miss * 100, annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=100,
            cbar_kws={"label": "missing (%)"}, ax=ax)
ax.set_title("Missing rate by position group (%)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(rotation=40, ha="right")
save_fig(fig, "combine_missing_by_position")
save_table(miss.round(3), "combine_missing_by_position")

In [ ]:
# 결측이 함께 일어나는가: 민첩성 3종 결측 조합별 인원
agil = ["three_cone", "short_shuttle", "bench_reps"]
cr[agil].isna().value_counts().rename("n").to_frame()

In [ ]:
# 결측 × 드래프트 티어 (players.csv에서 드래프트 정보만 사용)
pl = pd.read_csv(config.RAW_FILES["players"], usecols=["nfl_id", "draft_round"])
tier = pd.cut(pl["draft_round"], bins=[0, 3, 7], labels=["1-3R", "4-7R"])
pl["draft_tier"] = tier.cat.add_categories("UDFA").fillna("UDFA")

crp = cr.merge(pl[["nfl_id", "draft_tier"]], on="nfl_id", how="left")
miss_tier = (crp.groupby("draft_tier", observed=True)[DRILL]
                .apply(lambda d: d.isna().mean())
                .reindex(["1-3R", "4-7R", "UDFA"]))
display((miss_tier * 100).round(0))
save_table(miss_tier.round(3), "combine_missing_by_draft_tier")

## 3. 분포와 이상치
포지션 그룹마다 분포가 다르므로 **그룹을 나눠서** 본다. 이상치는 그룹 내 |z| > 3 기준으로 목록만 만든다(제거는 하지 않음).

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for ax, col in zip(axes.flat, MEAS):
    sns.kdeplot(data=cr, x=col, hue="combine_position", hue_order=POS,
                common_norm=False, fill=False, ax=ax, warn_singular=False)
    ax.set_title(col); ax.set_xlabel(""); ax.set_ylabel("")
    if ax is not axes.flat[0]:
        ax.get_legend().remove()
for ax in axes.flat[len(MEAS):]:
    ax.set_visible(False)
fig.suptitle("Traditional combine measures — density by position group", y=1.01)
fig.tight_layout()
save_fig(fig, "combine_measure_distributions")

In [ ]:
z = cr.groupby("combine_position")[MEAS].transform(lambda s: (s - s.mean()) / s.std())
rows, cols = np.nonzero((z.abs() > 3).to_numpy())
outliers = pd.DataFrame({
    "nfl_id": cr["nfl_id"].to_numpy()[rows],
    "position": cr["combine_position"].to_numpy()[rows],
    "measure": np.array(MEAS)[cols],
    "value": cr[MEAS].to_numpy()[rows, cols],
    "z_within_pos": z.to_numpy()[rows, cols],
}).sort_values("z_within_pos", key=abs, ascending=False)
print(f"그룹 내 |z|>3: {len(outliers)}건 / 선수 {outliers['nfl_id'].nunique()}명")
save_table(outliers.round(3), "combine_outliers_within_position", index=False)
outliers.head(15)

## 4. 측정치 간 상관: 전체 vs 포지션 그룹 내
**그룹 내 상관** = 각 값에서 포지션 평균을 뺀 뒤 계산한 상관(pooled within-group). 전체 상관이 그룹 내 상관보다 훨씬 강하면, 그 상관은 상당 부분 **포지션 차이가 만든 것**이다 → 모든 분석을 포지션 그룹 내에서 평가하는 근거.

In [ ]:
corr_all = cr[MEAS].corr()
centered = cr[MEAS] - cr.groupby("combine_position")[MEAS].transform("mean")
corr_within = centered.corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
kw = dict(vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".2f", annot_kws={"size": 7}, square=True)
sns.heatmap(corr_all, ax=axes[0], cbar=False, **kw);    axes[0].set_title("Correlation — overall")
sns.heatmap(corr_within, ax=axes[1], **kw);             axes[1].set_title("Correlation — within position group (group-mean removed)")
fig.tight_layout()
save_fig(fig, "combine_corr_overall_vs_within")

In [ ]:
# 전체와 그룹 내 상관 차이가 큰 쌍 상위 10
iu = np.triu_indices(len(MEAS), k=1)
pairs = pd.DataFrame({
    "a": np.array(MEAS)[iu[0]], "b": np.array(MEAS)[iu[1]],
    "r_all": corr_all.values[iu], "r_within": corr_within.values[iu],
})
pairs["gap"] = pairs["r_all"].abs() - pairs["r_within"].abs()
save_table(pairs.round(3), "combine_corr_pairs", index=False)
pairs.sort_values("gap", ascending=False).head(10).round(2)

## 5. combine_tracking 구조
10Hz 프레임 단위. 84MB라 한 번에 읽는다. 이후 분석은 `entity_type == "PLAYER"`만 사용(BALL은 비교용).

In [ ]:
ct = pd.read_csv(config.RAW_FILES["combine_tracking"])
ct["time"] = pd.to_datetime(ct["time"])
print(ct.shape)
display(ct["entity_type"].value_counts().to_frame("frames"))

ply = ct[ct["entity_type"] == "PLAYER"].copy()
ply = ply.merge(cr[["nfl_id", "combine_position"]], on="nfl_id", how="left")
print("PLAYER 프레임:", len(ply), "| 포지션 매칭 실패:", ply["combine_position"].isna().sum())

### 5-1. 트라이얼 키 확인
계획서 T1의 키는 `(nfl_id, drill_name, attempt)`. 같은 키가 **서로 다른 event_id(세션)** 에 걸쳐 있으면 두 트라이얼이 하나로 합쳐져 시간 간격이 비정상적으로 커진다.

In [ ]:
KEY_PLAN = ["nfl_id", "drill_name", "attempt"]
KEY_EVT  = ["nfl_id", "event_id", "drill_name", "attempt"]

n_plan = ply.groupby(KEY_PLAN).ngroups
n_evt  = ply.groupby(KEY_EVT).ngroups
print(f"계획서 키 트라이얼 수: {n_plan:,} | event_id 포함 키: {n_evt:,}")

multi = ply.groupby(KEY_PLAN)["event_id"].nunique().loc[lambda s: s > 1]
print(f"한 키에 event_id가 2개 이상인 경우: {len(multi)}건")
display(multi.to_frame("n_events"))

gap = ply.sort_values(KEY_PLAN + ["time"]).groupby(KEY_PLAN)["time"].diff().dt.total_seconds()
print("계획서 키 기준 프레임 간격 최대(초):", gap.max())

> **결정 필요**: 위 결과에서 중복이 있으면 트라이얼 키를 `KEY_EVT`(event_id 포함)로 바꾸고, 계획서 T1 키와 `config.py`에 반영한다. 이하 셀은 `TRIAL_KEY`를 사용한다.

In [ ]:
TRIAL_KEY = KEY_EVT if n_evt != n_plan else KEY_PLAN
print("TRIAL_KEY =", TRIAL_KEY)

### 5-2. 드릴별 트라이얼 수 · 선수 수

In [ ]:
trials = (ply.groupby(TRIAL_KEY + ["drill_type", "combine_position"])
             .agg(n_frames=("time", "size"),
                  t0=("time", "min"), t1=("time", "max"),
                  s_max=("s", "max"), a_max=("a", "max"),
                  max_gap=("time", lambda t: t.sort_values().diff().dt.total_seconds().max()))
             .reset_index())
trials["duration_s"] = (trials["t1"] - trials["t0"]).dt.total_seconds()
print("트라이얼 수:", len(trials))

drill_tbl = (trials.groupby(["drill_type", "drill_name"])
                   .agg(n_trials=("nfl_id", "size"), n_players=("nfl_id", "nunique"),
                        frames_median=("n_frames", "median"))
                   .sort_values(["drill_type", "n_trials"], ascending=[True, False]))
save_table(drill_tbl, "combine_trials_by_drill")
drill_tbl.groupby("drill_type").agg(n_drills=("n_trials", "size"),
                                    n_trials=("n_trials", "sum"))

In [ ]:
# 포지션 그룹 × 드릴 타입: 몇 명이 수행했나 (SKILL_DRILLS_LB 는 누가?)
pos_drill = (trials.groupby(["combine_position", "drill_type"])["nfl_id"].nunique()
                   .unstack(fill_value=0).reindex(POS))
display(pos_drill)
save_table(pos_drill, "combine_players_by_position_drilltype")

In [ ]:
# 선수당 40야드 시도 횟수 (최고 시도 vs 평균 집계 설계에 필요)
forty_tr = trials[trials["drill_type"] == "FORTY_YARD_DASH"]
forty_tr.groupby("nfl_id").size().value_counts().sort_index().rename_axis("attempts").to_frame("n_players")

## 6. 속도·가속 분포와 글리치 임계값
후보값: `SPEED_GLITCH_YDS = 12.5`, `ACCEL_GLITCH_YDS2 = 12.0`. 참고로 NFL 경기 최고속도 기록은 약 22~23 mph(≈ 10.8~11.2 yd/s).
PLAYER와 BALL을 나눠 보고, 임계값을 넘는 프레임·트라이얼이 얼마나 되는지 확인한다.

In [ ]:
qs = [0.5, 0.9, 0.99, 0.999, 0.9999, 1.0]
pct = pd.concat({
    ent: ct.loc[ct["entity_type"] == ent, ["s", "a"]].quantile(qs)
    for ent in ["PLAYER", "BALL"]
}, axis=1).round(2)
pct.index = [f"p{q*100:g}" for q in qs]
display(pct)
save_table(pct, "combine_speed_accel_percentiles")

S_TH, A_TH = config.SPEED_GLITCH_YDS, config.ACCEL_GLITCH_YDS2
over = pd.DataFrame({
    "frames_s_over": [(ply["s"] > S_TH).sum()],
    "frames_a_over": [(ply["a"] > A_TH).sum()],
    "trials_with_any": [((trials["s_max"] > S_TH) | (trials["a_max"] > A_TH)).sum()],
}, index=[f"PLAYER (s>{S_TH}, a>{A_TH})"])
over

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col, th in [(axes[0], "s", S_TH), (axes[1], "a", A_TH)]:
    for ent, c in [("PLAYER", "C0"), ("BALL", "C3")]:
        ax.hist(ct.loc[ct["entity_type"] == ent, col], bins=200, histtype="step",
                color=c, label=ent, density=True)
    ax.axvline(th, color="k", ls="--", lw=1, label=f"candidate threshold {th}")
    ax.set_yscale("log"); ax.set_xlabel(f"{col} ({'yd/s' if col == 's' else 'yd/s²'})")
    ax.legend()
axes[0].set_title("Speed distribution (log y)"); axes[1].set_title("Acceleration distribution (log y)")
fig.tight_layout()
save_fig(fig, "combine_speed_accel_hist")

In [ ]:
# 드릴 타입별 트라이얼 최고속도·최대가속 — 물리적으로 그럴듯한지
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
order = trials.groupby("drill_type")["s_max"].median().sort_values().index
sns.boxplot(data=trials, y="drill_type", x="s_max", order=order, ax=axes[0], fliersize=2)
sns.boxplot(data=trials, y="drill_type", x="a_max", order=order, ax=axes[1], fliersize=2)
axes[0].axvline(S_TH, color="k", ls="--", lw=1); axes[1].axvline(A_TH, color="k", ls="--", lw=1)
axes[0].set_title("Trial max speed (yd/s)"); axes[1].set_title("Trial max acceleration (yd/s²)")
axes[0].set_ylabel("")
fig.tight_layout()
save_fig(fig, "combine_trial_peaks_by_drill")

> **결정 필요**: 위 백분위·히스토그램을 근거로 컴바인 PLAYER 프레임에 쓸 임계값을 정하고 `config.py`에 반영한다. 넘는 프레임이 거의 없다면 "컴바인 PLAYER 데이터에는 글리치가 사실상 없음 — 글리치는 BALL에 집중"이 결론이 되며, 임계값은 경기 트래킹(Stage 3)용으로 01b에서 다시 검토한다.

## 7. 트라이얼 길이 이상치
드릴마다 정상 길이가 다르므로 **drill_name 내 분위수**로 본다. 후보 기준: 같은 drill_name 의 1백분위 미만 프레임 수, 또는 프레임 간격이 0.15초를 넘는 결손(10Hz 기준 0.1초).

주의: 시간 기반 드릴(40야드, 셔틀, 3-cone)은 **빠른 선수일수록 트라이얼이 짧다**. "짧다 = 잘렸다"가 아닐 수 있으므로 아래 마지막 셀의 속도 곡선으로 실제로 잘렸는지(최고속도 도달 전 종료 등) 확인하고 기준을 정한다.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.boxplot(data=trials, y="drill_type", x="duration_s", order=order, ax=ax, fliersize=2)
ax.set_xscale("log"); ax.set_title("Trial duration by drill type (s, log)"); ax.set_ylabel("")
save_fig(fig, "combine_trial_duration_by_drill")

In [ ]:
p01 = trials.groupby("drill_name")["n_frames"].transform(lambda s: s.quantile(0.01))
p50 = trials.groupby("drill_name")["n_frames"].transform("median")
trials["short_vs_drill"] = trials["n_frames"] < p01
trials["frac_of_median"] = trials["n_frames"] / p50
trials["has_gap"] = trials["max_gap"] > 0.15

flag_tbl = trials.groupby("drill_type")[["short_vs_drill", "has_gap"]].sum()
flag_tbl["n_trials"] = trials.groupby("drill_type").size()
display(flag_tbl)
save_table(flag_tbl, "combine_trial_length_flags")

trials.loc[trials["short_vs_drill"] | trials["has_gap"],
           TRIAL_KEY + ["drill_type", "n_frames", "duration_s", "frac_of_median", "max_gap"]] \
      .sort_values("frac_of_median").head(15)

In [ ]:
# 예시: 40야드 — 가장 짧은 트라이얼 3개 vs 중앙값 근처 3개의 속도 곡선
f = trials[trials["drill_type"] == "FORTY_YARD_DASH"].sort_values("n_frames")
mid = len(f) // 2
examples = pd.concat([f.head(3).assign(kind="short"), f.iloc[mid - 1: mid + 2].assign(kind="typical")])

fig, ax = plt.subplots(figsize=(9, 4))
for _, r in examples.iterrows():
    m = np.logical_and.reduce([ply[k] == r[k] for k in TRIAL_KEY])
    tr = ply.loc[m].sort_values("time")
    t = (tr["time"] - tr["time"].iloc[0]).dt.total_seconds()
    ax.plot(t, tr["s"], ls="-" if r["kind"] == "typical" else "--",
            label=f"{r['kind']} · {int(r['n_frames'])} frames")
ax.set_xlabel("time since trial start (s)"); ax.set_ylabel("s (yd/s)")
ax.set_title("40-yd dash speed curves: shortest vs typical trials"); ax.legend(fontsize=8)
save_fig(fig, "combine_forty_short_vs_typical")

## 8. 요약 · 결정 사항 (→ `reports/eda.md` 컴바인 절)

**확인한 사실**
- (1절) 행 수 · 포지션 구성: …
- (2절) 결측: 3-cone/shuttle/bench 결측은 주로 … 포지션 / 드래프트 티어별 차이: …
- (4절) 전체 vs 그룹 내 상관: …
- (5절) 트라이얼 키: …
- (6절) PLAYER 속도·가속 최댓값 …, BALL …

**결정 사항 (config.py · 계획서에 반영)**
| 항목 | 후보 | 결정 | 근거 |
|---|---|---|---|
| 트라이얼 키 | `(nfl_id, drill_name, attempt)` | | 5-1 |
| `SPEED_GLITCH_YDS` | 12.5 | | 6절 |
| `ACCEL_GLITCH_YDS2` | 12.0 | | 6절 |
| truncated 기준 | drill 내 p1 미만 / 간격 > 0.15s | | 7절 |

**Stage 2·4로 넘길 메모**
- …